In [88]:
import pandas as pd

df = pd.read_csv("../data/processed/audible_clean.csv")

df_advance = pd.read_csv(
    "../data/processed/audible_advanced_clean.csv"
)

print("Dataset 1 Shape:", df.shape)
print("Dataset 2 Shape:", df_advance.shape)

Dataset 1 Shape: (5439, 5)
Dataset 2 Shape: (4293, 11)


In [89]:
print("Dataset 1 Columns:")
print(df.columns.tolist())

print("\nDataset 2 Columns:")
print(df_advance.columns.tolist())

Dataset 1 Columns:
['Book Name', 'Author', 'Rating', 'Number of Reviews', 'Price']

Dataset 2 Columns:
['Book Name', 'Author', 'Rating', 'Number of Reviews', 'Price', 'Description', 'Listening Time', 'Ranks and Genre', 'Listening_Time_Minutes', 'Genres', 'Content']


In [90]:
df_advance[[
    "Book Name",
    "Author",
    "Description",
    "Ranks and Genre"
]].head(5)

,Book Name,Author,Description,Ranks and Genre
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,"Over the past three years, Jay Shetty has beco...",",#1 in Audible Audiobooks & Originals (See Top..."
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,Brought to you by Penguin.,",#2 in Audible Audiobooks & Originals (See Top..."
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,"In this generation-defining self-help guide, a...",",#3 in Audible Audiobooks & Originals (See Top..."
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,Brought to you by Penguin.,",#5 in Audible Audiobooks & Originals (See Top..."
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,"Stop going through life, Start growing throug...",",#6 in Audible Audiobooks & Originals (See Top..."


In [91]:
# Display complete genre/rank information for 10 books
for value in df_advance["Ranks and Genre"].head(10):
    print(value)
    print("-" * 80)

,#1 in Audible Audiobooks & Originals (See Top 100 in Audible Audiobooks & Originals),#1 in Personal Success,#1 in Stress Management,#2 in Society & Culture (Books)
--------------------------------------------------------------------------------
,#2 in Audible Audiobooks & Originals (See Top 100 in Audible Audiobooks & Originals),#1 in Meditation (Audible Audiobooks & Originals),#1 in Self-Esteem,#2 in Personal Success
--------------------------------------------------------------------------------
,#3 in Audible Audiobooks & Originals (See Top 100 in Audible Audiobooks & Originals),#3 in Personal Success,#16 in Personal Development & Self-Help
--------------------------------------------------------------------------------
,#5 in Audible Audiobooks & Originals (See Top 100 in Audible Audiobooks & Originals),#1 in Psychology (Audible Audiobooks & Originals),#3 in Stress Management,#4 in Personal Success
--------------------------------------------------------------------------------
,#

In [92]:
import re

def extract_genres(rank_genre):
    if pd.isna(rank_genre):
        return []

    genres = re.findall(r"#\d+\s+in\s+([^,]+)", rank_genre)

    # Remove the overall Audible ranking
    genres = [
        genre.strip()
        for genre in genres
        if "Audible Audiobooks & Originals" not in genre
    ]

    return genres

df_advance["Genres"] = df_advance["Ranks and Genre"].apply(extract_genres)

df_advance[["Book Name", "Genres"]].head(10)

,Book Name,Genres
0,Think Like a Monk: The Secret of How to Harnes...,"[Personal Success, Stress Management, Society ..."
1,Ikigai: The Japanese Secret to a Long and Happ...,"[Self-Esteem, Personal Success]"
2,The Subtle Art of Not Giving a F*ck: A Counter...,"[Personal Success, Personal Development & Self..."
3,Atomic Habits: An Easy and Proven Way to Build...,"[Stress Management, Personal Success]"
4,Life's Amazing Secrets: How to Find Balance an...,"[Literary Essays, Essays (Books), Spiritualism]"
5,Sapiens,"[History of Civilization, Anthropology (Books)]"
6,Extraordinary Leadership,"[Leadership, Business Careers]"
7,The Intelligent Investor Rev Ed.,"[Personal Finance, Investing & Trading, Analys..."
8,Rich Dad Poor Dad: What the Rich Teach Their K...,"[Personal Finance, Investing & Trading, Person..."
9,The 5AM Club: Own Your Morning. Elevate Your L...,"[Sleep Disorders, Personal Success, Healthy Li..."


In [93]:
# Combine Description and Genres
df_advance["Content"] = (
    df_advance["Description"].fillna("") + " " +
    df_advance["Genres"].apply(lambda x: " ".join(x))
)

# Display the combined content
df_advance[["Book Name", "Content"]].head(5)

,Book Name,Content
0,Think Like a Monk: The Secret of How to Harnes...,"Over the past three years, Jay Shetty has beco..."
1,Ikigai: The Japanese Secret to a Long and Happ...,Brought to you by Penguin. Self-Esteem Persona...
2,The Subtle Art of Not Giving a F*ck: A Counter...,"In this generation-defining self-help guide, a..."
3,Atomic Habits: An Easy and Proven Way to Build...,Brought to you by Penguin. Stress Management P...
4,Life's Amazing Secrets: How to Find Balance an...,"Stop going through life, Start growing throug..."


In [94]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Generate TF-IDF matrix
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

tfidf_matrix = tfidf.fit_transform(
    df_advance["Content"]
)

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

# Calculate cosine similarity
cosine_sim = cosine_similarity(
    tfidf_matrix,
    tfidf_matrix
)

print("Cosine Similarity Shape:", cosine_sim.shape)

print("\nModel features regenerated successfully!")


TF-IDF Matrix Shape: (4293, 10000)
Cosine Similarity Shape: (4293, 4293)

Model features regenerated successfully!


In [95]:

def recommend_book(book_name, cosine_sim=cosine_sim):

    # Find selected book
    matches = np.flatnonzero(
        df_advance["Book Name"].eq(book_name).to_numpy()
    )

    if len(matches) == 0:
        print("Book not found in the dataset.")
        return

    idx = int(matches[0])

    # Get similarity scores
    similarity_scores = cosine_sim[idx]

    # Sort by similarity
    book_indices = np.argsort(similarity_scores)[::-1]

    recommendations_list = []
    selected_title = book_name.strip().casefold()

    for i in book_indices:

        candidate_title = str(
            df_advance.iloc[i]["Book Name"]
        ).strip().casefold()

        # Exclude selected title and duplicate titles
        if candidate_title == selected_title:
            continue

        # Exclude zero similarity
        if similarity_scores[i] <= 0:
            continue

        # Avoid duplicate recommendations
        if candidate_title in [
            item["Book Name"].strip().casefold()
            for item in recommendations_list
        ]:
            continue

        recommendations_list.append({
            "Book Name": df_advance.iloc[i]["Book Name"],
            "Author": df_advance.iloc[i]["Author"],
            "Rating": df_advance.iloc[i]["Rating"],
            "Genres": df_advance.iloc[i]["Genres"],
            "Similarity Score": round(
                float(similarity_scores[i]) * 100, 2
            )
        })

        if len(recommendations_list) == 5:
            break

    if not recommendations_list:
        print("No meaningful recommendations found.")
        return

    return pd.DataFrame(recommendations_list)


In [96]:
import numpy as np
# Test the recommendation system

book_name = df_advance["Book Name"].iloc[0]
print("Selected Book:", book_name)
recommendations = recommend_book(book_name)

print("\nTop 5 Recommended Books:")
display(recommendations)


Selected Book: Think Like a Monk: The Secret of How to Harness the Power of Positivity and Be Happy Now

Top 5 Recommended Books:


,Book Name,Author,Rating,Genres,Similarity Score
0,"The Classic Tales Podcast, Season Five",Agatha Christie,NaN,"[Caribbean & Latin American Literature, English]",17.04
1,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4.6,"[Stress Management, Personal Success]",16.30
2,The Facebook Effect: The Inside Story of the C...,David Kirkpatrick,4.4,"[Social Media, Technology & Society, E-Commerce]",16.18
3,Ten Arguments For Deleting Your Social Media A...,Jaron Lanier,4.4,"[Social Media, Popular Culture Studies, Techno...",14.94
4,Social Media Marketing Workbook 2020,Jason McDonald PhD,4.5,"[Social Media, Internet & Social Media (Books)]",14.29


In [97]:

# Test invalid book title

recommendations = recommend_book("Unknown Book XYZ")


Book not found in the dataset.


Evaluation of Recommandation systen

In [98]:

# Evaluate Top 5 recommendations using Cosine Similarity

book_name = df_advance["Book Name"].iloc[0]

indices = pd.Series(
    df_advance.index,
    index=df_advance["Book Name"]
).drop_duplicates()

idx = indices[book_name]

similarity_scores = list(enumerate(cosine_sim[idx]))

similarity_scores = sorted(
    similarity_scores,
    key=lambda x: x[1],
    reverse=True
)

similarity_scores = similarity_scores[1:6]

book_indices = [i[0] for i in similarity_scores]
scores = [i[1] for i in similarity_scores]

evaluation_results = df_advance.iloc[book_indices][
    ["Book Name", "Author", "Rating", "Genres"]
].copy()

evaluation_results["Similarity Score"] = scores

print("Selected Book:", book_name)

display(evaluation_results)


Selected Book: Think Like a Monk: The Secret of How to Harness the Power of Positivity and Be Happy Now


,Book Name,Author,Rating,Genres,Similarity Score
1865,"The Classic Tales Podcast, Season Five",Agatha Christie,NaN,"[Caribbean & Latin American Literature, English]",0.170374
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4.6,"[Stress Management, Personal Success]",0.162975
1295,The Facebook Effect: The Inside Story of the C...,David Kirkpatrick,4.4,"[Social Media, Technology & Society, E-Commerce]",0.161836
448,Ten Arguments For Deleting Your Social Media A...,Jaron Lanier,4.4,"[Social Media, Popular Culture Studies, Techno...",0.149434
1043,Social Media Marketing Workbook 2020,Jason McDonald PhD,4.5,"[Social Media, Internet & Social Media (Books)]",0.142885


In [99]:

# Calculate Average Similarity Score

average_similarity = evaluation_results["Similarity Score"].mean()

print("Average Similarity Score:", round(average_similarity, 4))
print("Average Similarity Percentage:", round(average_similarity * 100, 2), "%")


Average Similarity Score: 0.1575
Average Similarity Percentage: 15.75 %


In [100]:

# Select a book with a known title

book_name = "Atomic Habits: An Easy and Proven Way to Build Good Habits and Break Bad Ones"

recommendations = recommend_book(book_name)

display(recommendations)


,Book Name,Author,Rating,Genres,Similarity Score
0,The Stress Solution: The 4 Steps to Reset Your...,Dr Rangan Chatterjee,4.7,[Stress Management],87.86
1,The Antidote: Happiness for People Who Can't S...,Oliver Burkeman,4.5,[Personal Success],74.49
2,Surrounded by Idiots: The Four Types of Human ...,Thomas Erikson,4.5,[Personal Success],74.49
3,Happy: Why More or Less Everything Is Absolute...,Derren Brown,4.4,[Personal Success],74.49
4,"Die Trying: Jack Reacher, Book 2",Lee Child,4.4,[],57.17


In [101]:

# Calculate Precision@5

relevant_books = 4
k = 5

precision_at_5 = relevant_books / k

print("Precision@5:", round(precision_at_5, 2))
print("Precision@5 Percentage:", round(precision_at_5 * 100, 2), "%")


Precision@5: 0.8
Precision@5 Percentage: 80.0 %


In [102]:
# Test recommendations for multiple audiobooks

test_books = df_advance["Book Name"].dropna().drop_duplicates().sample(
    5, random_state=42
)

for book in test_books:
    print("\nSelected Book:", book)
    print("-" * 80)

    recommendations = recommend_book(book)

    display(recommendations)


Selected Book: The Mixer: The Story of Premier League Tactics, from Route One to False Nines
--------------------------------------------------------------------------------


,Book Name,Author,Rating,Genres,Similarity Score
0,The Soccer Mystery: The Boxcar Children Myster...,Gertrude Chandler Warner,5.0,[],19.62
1,"Coaching for Performance, Fourth Edition: GROW...",John Whitmore,4.5,"[Mentoring & Coaching, Leadership]",16.69
2,Learn Hindi - Level 1: Introduction to Hindi: ...,Innovative Language Learning LLC,NaN,[],14.48
3,101 Amazing Facts About the Royal Family,Jack Goldstein,2.6,[Biographies of Royalty],13.81
4,The Quality of Madness,Tim Rich,4.8,"[Sports Biographies, Sports (Books)]",12.18



Selected Book: Leadership on the Line (Revised): Staying Alive Through the Dangers of Change
--------------------------------------------------------------------------------
No meaningful recommendations found.


None


Selected Book: The Disaster Artist
--------------------------------------------------------------------------------
No meaningful recommendations found.


None


Selected Book: The Gravity Between Us
--------------------------------------------------------------------------------
No meaningful recommendations found.


None


Selected Book: 101 Animals Stories
--------------------------------------------------------------------------------
No meaningful recommendations found.


None

In [103]:

print("Cosine Similarity Shape:", cosine_sim.shape)
print("Data Type:", type(cosine_sim))
print("Selected Row Shape:", cosine_sim[0].shape)


Cosine Similarity Shape: (4293, 4293)
Data Type: <class 'numpy.ndarray'>
Selected Row Shape: (4293,)


In [104]:

book_name = test_books.iloc[0]

indices = pd.Series(
    range(len(df_advance)),
    index=df_advance["Book Name"]
).drop_duplicates()

idx = indices[book_name]

print("Selected Book:", book_name)
print("Book Index:", idx)
print("Index Type:", type(idx))
print("Similarity Row Shape:", cosine_sim[idx].shape)


Selected Book: The Mixer: The Story of Premier League Tactics, from Route One to False Nines
Book Index: 1307
Index Type: <class 'numpy.int64'>
Similarity Row Shape: (4293,)


In [105]:
import numpy as np
book_name = test_books.iloc[0]

matches = np.flatnonzero(
    df_advance["Book Name"].eq(book_name).to_numpy()
)

print("Selected Book:", book_name)
print("Matching Positions:", matches)
print("Number of Matches:", len(matches))

if len(matches) > 0:
    idx = int(matches[0])
    print("Selected Index:", idx)
    print("Index Type:", type(idx))
    print("Similarity Row Shape:", cosine_sim[idx].shape)

Selected Book: The Mixer: The Story of Premier League Tactics, from Route One to False Nines
Matching Positions: [1307]
Number of Matches: 1
Selected Index: 1307
Index Type: <class 'int'>
Similarity Row Shape: (4293,)


In [106]:
similarity_row = cosine_sim[idx]

print("Data Type:", similarity_row.dtype)
print("Shape:", similarity_row.shape)

print("\nNaN Values:", np.isnan(similarity_row).sum())
print("Infinite Values:", np.isinf(similarity_row).sum())

print("\nMinimum:", np.min(similarity_row))
print("Maximum:", np.max(similarity_row))

print("\nFirst 10 Values:")
print(similarity_row[:10])

print("\nUnique Values Count:", len(np.unique(similarity_row)))

Data Type: float64
Shape: (4293,)

NaN Values: 0
Infinite Values: 0

Minimum: 0.0
Maximum: 1.0000000000000002

First 10 Values:
[0.02300503 0.         0.03592887 0.         0.01871291 0.01513249
 0.         0.         0.0451069  0.01804742]

Unique Values Count: 987


In [107]:
similarity_scores = list(enumerate(cosine_sim[idx]))

print("Similarity Scores Type:", type(similarity_scores))
print("First Item:", similarity_scores[0])
print("First Item Type:", type(similarity_scores[0]))

print("\nSecond Element Type:", type(similarity_scores[0][1]))
print("Second Element Shape:", np.shape(similarity_scores[0][1]))

print("\nChecking All Scores:")

for i, score in similarity_scores:
    if np.ndim(score) != 0:
        print("Non-scalar score found!")
        print("Index:", i)
        print("Score:", score)
        print("Shape:", np.shape(score))
        break
else:
    print("All similarity scores are scalar values.")

Similarity Scores Type: <class 'list'>
First Item: (0, np.float64(0.02300502502938792))
First Item Type: <class 'tuple'>

Second Element Type: <class 'numpy.float64'>
Second Element Shape: ()

Checking All Scores:
All similarity scores are scalar values.


In [108]:
print("Total Books:", len(df_advance))

print("\nEmpty Content:")
print(df_advance["Content"].isna().sum())

print("\nBlank Content:")
print(
    df_advance["Content"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("\nContent Length Statistics:")
print(df_advance["Content"].fillna("").str.len().describe())

print("\nBooks with Shortest Content:")
display(
    df_advance.assign(
        Content_Length=df_advance["Content"].fillna("").str.len()
    ).nsmallest(10, "Content_Length")[
        ["Book Name", "Genres", "Content", "Content_Length"]
    ]
)

Total Books: 4293

Empty Content:
0

Blank Content:
1056

Content Length Statistics:
count    4293.000000
mean      223.697181
std       227.875991
min         1.000000
25%        21.000000
50%       208.000000
75%       316.000000
max      2226.000000
Name: Content, dtype: float64

Books with Shortest Content:


,Book Name,Genres,Content,Content_Length
48,The Hard Thing About Hard Things: Building a B...,[],,1
51,"Men Are from Mars, Women Are from Venus: The C...",[],,1
57,Hit Refresh,[],,1
58,A Suitable Boy (Dramatised),[],,1
59,Take Control of Your Life: How to Silence Fear...,[],,1
60,The 10X Rule: The Only Difference Between Succ...,[],,1
61,A Game of Thrones: Book 1 of A Song of Ice and...,[],,1
144,Breath: The New Science of a Lost Art,[],,1
145,Origin,[],,1
146,Chinta Chhodo Sukh Se Jiyo,[],,1


In [109]:
# Replace missing descriptions and genres with empty strings
df_advance["Description"] = df_advance["Description"].fillna("").astype(str)
df_advance["Genres"] = df_advance["Genres"].apply(
    lambda x: " ".join(x) if isinstance(x, list) else str(x)
)

# Build improved recommendation content
df_advance["Content"] = (
    df_advance["Description"] + " " +
    df_advance["Genres"] + " " +
    df_advance["Book Name"].fillna("") + " " +
    df_advance["Author"].fillna("")
).str.strip()

# Check updated content
print("Total Books:", len(df_advance))

print(
    "Blank Content:",
    df_advance["Content"].str.strip().eq("").sum()
)

print("\nUpdated Content Examples:")
display(
    df_advance[
        ["Book Name", "Author", "Genres", "Content"]
    ].head(10)
)

Total Books: 4293
Blank Content: 0

Updated Content Examples:


,Book Name,Author,Genres,Content
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,Personal Success Stress Management Society & C...,"Over the past three years, Jay Shetty has beco..."
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,Self-Esteem Personal Success,Brought to you by Penguin. Self-Esteem Persona...
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,Personal Success Personal Development & Self-Help,"In this generation-defining self-help guide, a..."
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,Stress Management Personal Success,Brought to you by Penguin. Stress Management P...
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,Literary Essays Essays (Books) Spiritualism,"Stop going through life, Start growing throug..."
5,Sapiens,Yuval Noah Harari,History of Civilization Anthropology (Books),Fire gave us power. Farming made us hungry for...
6,Extraordinary Leadership,Robin Sharma,Leadership Business Careers,Leadership Business Careers Extraordinary Lead...
7,The Intelligent Investor Rev Ed.,Benjamin Graham,Personal Finance Investing & Trading Analysis ...,The Classic Text Annotated to Update Graham's ...
8,Rich Dad Poor Dad: What the Rich Teach Their K...,Robert T. Kiyosaki,Personal Finance Investing & Trading Personal ...,Rich Dad Poor Dad will…. Personal Finance Inve...
9,The 5AM Club: Own Your Morning. Elevate Your L...,Robin Sharma,Sleep Disorders Personal Success Healthy Livin...,Legendary leadership and elite performance exp...


In [110]:
import pandas as pd

df_advance = pd.read_csv(
    "../data/processed/audible_advanced_clean.csv"
)

print("Dataset Loaded Successfully!")
print("Shape:", df_advance.shape)
print("Columns:", df_advance.columns.tolist())

Dataset Loaded Successfully!
Shape: (4293, 11)
Columns: ['Book Name', 'Author', 'Rating', 'Number of Reviews', 'Price', 'Description', 'Listening Time', 'Ranks and Genre', 'Listening_Time_Minutes', 'Genres', 'Content']


In [111]:
df_advance = pd.read_csv(
    "../data/processed/audible_advanced_clean.csv"
)

In [112]:

import pandas as pd
import re

def extract_genres(value):
    if pd.isna(value):
        return ""

    # Split the ranking information
    entries = str(value).split(",")

    genres = []

    for entry in entries:
        entry = entry.strip()

        # Extract genre names after ranking numbers
        match = re.search(r"#\d+\s+in\s+(.+)", entry)

        if match:
            genre = match.group(1).strip()

            # Exclude overall Audible ranking
            if "Audible Audiobooks & Originals" not in genre:
                genres.append(genre)

    return " ".join(dict.fromkeys(genres))


# Recreate Genres
df_advance["Genres"] = df_advance["Ranks and Genre"].apply(
    extract_genres
)

# Rebuild Content
df_advance["Content"] = (
    df_advance["Book Name"].fillna("").astype(str) + " " +
    df_advance["Author"].fillna("").astype(str) + " " +
    df_advance["Description"].fillna("").astype(str) + " " +
    df_advance["Genres"]
)

# Validate
print("Total Books:", len(df_advance))
print("Blank Genres:", df_advance["Genres"].eq("").sum())
print("Blank Content:", df_advance["Content"].str.strip().eq("").sum())

display(
    df_advance[["Book Name", "Genres", "Content"]].head(10)
)


Total Books: 4293
Blank Genres: 2384
Blank Content: 0


,Book Name,Genres,Content
0,Think Like a Monk: The Secret of How to Harnes...,Personal Success Stress Management Society & C...,Think Like a Monk: The Secret of How to Harnes...
1,Ikigai: The Japanese Secret to a Long and Happ...,Self-Esteem Personal Success,Ikigai: The Japanese Secret to a Long and Happ...
2,The Subtle Art of Not Giving a F*ck: A Counter...,Personal Success Personal Development & Self-Help,The Subtle Art of Not Giving a F*ck: A Counter...
3,Atomic Habits: An Easy and Proven Way to Build...,Stress Management Personal Success,Atomic Habits: An Easy and Proven Way to Build...
4,Life's Amazing Secrets: How to Find Balance an...,Literary Essays Essays (Books) Spiritualism,Life's Amazing Secrets: How to Find Balance an...
5,Sapiens,History of Civilization Anthropology (Books),Sapiens Yuval Noah Harari Fire gave us power. ...
6,Extraordinary Leadership,Leadership Business Careers,Extraordinary Leadership Robin Sharma Leaders...
7,The Intelligent Investor Rev Ed.,Personal Finance Investing & Trading Analysis ...,The Intelligent Investor Rev Ed. Benjamin Grah...
8,Rich Dad Poor Dad: What the Rich Teach Their K...,Personal Finance Investing & Trading Personal ...,Rich Dad Poor Dad: What the Rich Teach Their K...
9,The 5AM Club: Own Your Morning. Elevate Your L...,Sleep Disorders Personal Success Healthy Livin...,The 5AM Club: Own Your Morning. Elevate Your L...


In [113]:

# Save updated dataset with Genres and Content

df_advance.to_csv(
    "../data/processed/audible_advanced_clean.csv",
    index=False
)

print("Updated dataset saved successfully!")

# Verify saved dataset
df_check = pd.read_csv(
    "../data/processed/audible_advanced_clean.csv"
)

print("\nSaved Dataset Shape:", df_check.shape)
print("Genres Column Exists:", "Genres" in df_check.columns)
print("Content Column Exists:", "Content" in df_check.columns)

print("Blank Genres:", df_check["Genres"].fillna("").eq("").sum())
print("Blank Content:", df_check["Content"].fillna("").str.strip().eq("").sum())


Updated dataset saved successfully!

Saved Dataset Shape: (4293, 11)
Genres Column Exists: True
Content Column Exists: True
Blank Genres: 2384
Blank Content: 0


In [114]:

# Check content availability for books without genres

print("Books with Genres:", 
      (df_advance["Genres"].str.strip() != "").sum())

print("Books without Genres:", 
      (df_advance["Genres"].str.strip() == "").sum())

print("\nDescription availability:")
print(
    df_advance.loc[
        df_advance["Genres"].str.strip() == "",
        "Description"
    ].notna().value_counts()
)

# Display sample books without genres
display(
    df_advance.loc[
        df_advance["Genres"].str.strip() == "",
        ["Book Name", "Author", "Description", "Genres", "Content"]
    ].head(5)
)


Books with Genres: 1909
Books without Genres: 2384

Description availability:
Description
True     1328
False    1056
Name: count, dtype: int64


,Book Name,Author,Description,Genres,Content
48,The Hard Thing About Hard Things: Building a B...,Ben Horowitz,NaN,,The Hard Thing About Hard Things: Building a B...
49,The Bhagavad Gita,Eknath Easwaran,"The Bhagavad Gita, ""The Song of the Lord"", is ...",,The Bhagavad Gita Eknath Easwaran The Bhagavad...
50,To Kill a Mockingbird,Harper Lee,Brought to you by Penguin.,,To Kill a Mockingbird Harper Lee Brought to yo...
51,"Men Are from Mars, Women Are from Venus: The C...",John Gray,NaN,,"Men Are from Mars, Women Are from Venus: The C..."
52,Trading in the Zone: Master the Market with Co...,Mark Douglas,Oops!\nIt's rush hour and traffic is piling up...,,Trading in the Zone: Master the Market with Co...


In [115]:
# Check feature columns before building weighted TF-IDF

feature_columns = ["Book Name", "Description", "Genres"]

print(df_advance[feature_columns].head(5).to_string(index=False))

                                                                               Book Name                                                                                                                                                                                                                                                                                                                                                                                                                                               Description                                                       Genres
Think Like a Monk: The Secret of How to Harness the Power of Positivity and Be Happy Now Over the past three years, Jay Shetty has become one of the world’s most popular influencers. One of his clips was the most watched video on Facebook last year, with more than 360 million views. His social media following totals more than 32 million, he has produced more than 400 viral videos which have amassed more t

In [116]:

# Import required libraries

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Required libraries imported successfully.")


Required libraries imported successfully.


TF-IDF stands for Term Frequency–Inverse Document Frequency. It is a text vectorization technique used in my audiobook recommendation system. It converts textual information into numerical vectors by assigning importance to words. Term Frequency measures how frequently a word occurs in a document, while Inverse Document Frequency measures how rare that word is across the dataset. These numerical representations are then used with cosine similarity to identify similar audiobooks.

In [125]:
# Final TF-IDF vectorizers

title_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

description_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

genre_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=3000
)

# Generate feature matrices

title_matrix = title_vectorizer.fit_transform(
    df_advance["Book Name"].fillna("")
)

description_matrix = description_vectorizer.fit_transform(
    df_advance["Description"].fillna("")
)

genre_matrix = genre_vectorizer.fit_transform(
    df_advance["Genres"].fillna("")
)

print("Title Matrix:", title_matrix.shape)
print("Description Matrix:", description_matrix.shape)
print("Genre Matrix:", genre_matrix.shape)

Title Matrix: (4293, 5000)
Description Matrix: (4293, 10000)
Genre Matrix: (4293, 780)


In [126]:
# Calculate cosine similarity for each feature

title_similarity = cosine_similarity(title_matrix)

description_similarity = cosine_similarity(description_matrix)

genre_similarity = cosine_similarity(genre_matrix)

print("Title Similarity:", title_similarity.shape)
print("Description Similarity:", description_similarity.shape)
print("Genre Similarity:", genre_similarity.shape)

Title Similarity: (4293, 4293)
Description Similarity: (4293, 4293)
Genre Similarity: (4293, 4293)


In [124]:
# Combine similarities using weighted scores

weighted_similarity = (
    0.2 * title_similarity +
    0.6 * description_similarity +
    0.2 * genre_similarity
)

print("Weighted Similarity:", weighted_similarity.shape)

Weighted Similarity: (4293, 4293)


In [128]:
# Final Book Recommendation Function

def recommend_book_final(book_title, top_n=5):

    book_indices = df_advance.index[
        df_advance["Book Name"] == book_title
    ].tolist()

    if not book_indices:
        print("Book not found in the dataset.")
        return pd.DataFrame()

    selected_index = book_indices[0]

    scores = weighted_similarity[selected_index]

    results = df_advance.copy()
    results["Similarity Score"] = scores * 100

    results = results[results["Book Name"] != book_title]
    results = results[results["Similarity Score"] > 0]

    results = results.sort_values(
        "Similarity Score",
        ascending=False
    )

    results = results.drop_duplicates(
        subset="Book Name",
        keep="first"
    )

    return results[
        ["Book Name", "Author", "Rating", "Genres", "Similarity Score"]
    ].head(top_n).round({"Similarity Score": 2})

In [129]:
# Test the final recommendation system

recommend_book_final("The Mixer: The Story of Premier League Tactics, from Route One to False Nines")

,Book Name,Author,Rating,Genres,Similarity Score
987,Leadership Strategy and Tactics: Field Manual,Jocko Willink,4.7,Time Management & Productivity Forecasting & S...,10.45
2212,101 Amazing Facts About the Royal Family,Jack Goldstein,2.6,Biographies of Royalty,9.67
1967,Learn Hindi - Level 1: Introduction to Hindi: ...,Innovative Language Learning LLC,NaN,,9.42
974,The Quality of Madness,Tim Rich,4.8,Sports Biographies Sports (Books),9.07
1251,The Third Door: The Wild Quest to Uncover How ...,Alex Banayan,4.7,Mentoring & Coaching Personal Success in Business,8.94


In [130]:
recommend_book_final("101 Animals Stories")

,Book Name,Author,Rating,Genres,Similarity Score
2537,My Family and Other Animals,Nigel Davenport,4.6,Environmentalists & Naturalists Biographies An...,10.40
609,101 Panchatantra Stories,Om Books International,4.1,Fiction Short Stories Short Stories (Books),9.97
1193,101 Panchtantra Stories,Dreamland Publications,4.3,Fairy Tales Literature & Fiction for Children ...,9.53
988,Byomkesh Bakshi Stories,Saradindu Bandopadhyay,4.4,Traditional Detective Mysteries,8.71
3381,101 Power Thoughts,Louise L. Hay,4.4,,6.75


In [131]:
# Evaluate recommendations for multiple audiobooks

test_books = [
    "The Mixer: The Story of Premier League Tactics, from Route One to False Nines",
    "101 Animals Stories",
    "Think Like a Monk: The Secret of How to Harness the Power of Positivity and Be Happy Now",
    "The Third Door: The Wild Quest to Uncover How the World's Most Successful People Launched Their Careers",
    "The Quality of Madness"
]

for book in test_books:
    print("\nSelected Book:", book)
    print("=" * 100)

    display(recommend_book_final(book))


Selected Book: The Mixer: The Story of Premier League Tactics, from Route One to False Nines


,Book Name,Author,Rating,Genres,Similarity Score
987,Leadership Strategy and Tactics: Field Manual,Jocko Willink,4.7,Time Management & Productivity Forecasting & S...,10.45
2212,101 Amazing Facts About the Royal Family,Jack Goldstein,2.6,Biographies of Royalty,9.67
1967,Learn Hindi - Level 1: Introduction to Hindi: ...,Innovative Language Learning LLC,NaN,,9.42
974,The Quality of Madness,Tim Rich,4.8,Sports Biographies Sports (Books),9.07
1251,The Third Door: The Wild Quest to Uncover How ...,Alex Banayan,4.7,Mentoring & Coaching Personal Success in Business,8.94



Selected Book: 101 Animals Stories


,Book Name,Author,Rating,Genres,Similarity Score
2537,My Family and Other Animals,Nigel Davenport,4.6,Environmentalists & Naturalists Biographies An...,10.40
609,101 Panchatantra Stories,Om Books International,4.1,Fiction Short Stories Short Stories (Books),9.97
1193,101 Panchtantra Stories,Dreamland Publications,4.3,Fairy Tales Literature & Fiction for Children ...,9.53
988,Byomkesh Bakshi Stories,Saradindu Bandopadhyay,4.4,Traditional Detective Mysteries,8.71
3381,101 Power Thoughts,Louise L. Hay,4.4,,6.75



Selected Book: Think Like a Monk: The Secret of How to Harness the Power of Positivity and Be Happy Now


,Book Name,Author,Rating,Genres,Similarity Score
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4.6,Stress Management Personal Success,14.46
1309,Biofeedback and Mindfulness in Everyday Life: ...,Inna Khazan PhD,4.8,Stress Management,14.36
536,Remnants of a Separation: A History of the Par...,Aanchal Malhotra,4.6,Society & Culture (Books),13.82
592,Discover Your Destiny (Hindi): With the Monk w...,Robin Sharma,4.2,Personal Success,12.70
850,Happy: Why More or Less Everything Is Absolute...,Derren Brown,4.4,Personal Success,12.34



Selected Book: The Third Door: The Wild Quest to Uncover How the World's Most Successful People Launched Their Careers


,Book Name,Author,Rating,Genres,Similarity Score
385,Murder on the Orient Express: An Audible Origi...,Agatha Christie,NaN,Amateur Sleuth Mysteries Cosy Mysteries Tradit...,21.04
585,Normal People,Sally Rooney,4.3,Literary Fiction,18.98
2310,Nice Girls Don't Get the Corner Office: Uncons...,Lois P. Frankel,4.5,Women in Business Personal Success,18.59
784,"Coaching for Performance, Fourth Edition: GROW...",John Whitmore,4.5,Mentoring & Coaching Leadership,18.31
686,Anything You Want: 40 Lessons for a New Kind o...,Derek Sivers,4.7,Personal Success in Business,18.16



Selected Book: The Quality of Madness


,Book Name,Author,Rating,Genres,Similarity Score
1583,J. Krishnamurti,Pupul Jayakar,4.5,,18.43
1704,The Captain Class: The Hidden Force Behind the...,Sam Walker,4.5,Sociology of Sports Coaching Sports Psychology,17.57
2144,The Rider,Tim Krabbé,4.7,Sports Fiction,16.88
2324,"Fighting for You: Bragan University Series, Bo...",Gianna Gabriela,5.0,Sports Fiction,16.13
1974,The Rise of Superman: Decoding the Science of ...,Steven Kotler,4.4,Extreme Sports Sports (Books),15.73


In [133]:
import joblib

joblib.dump(
    weighted_similarity,
    "../data/processed/weighted_similarity.pkl"
)

print("Final recommendation model saved successfully!")

Final recommendation model saved successfully!
